# A1.10 · Jailbreaks, model inversion and extraction

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.9 · The injection surface: direct and indirect prompt injection](https://spbreed.github.io/cyber-commons/lessons/A1.9.html)**.

| | |
|---|---|
| Open-source tooling | garak, Presidio |
| Open-weight models | Llama Guard 4 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Everything at the model layer gets called "a jailbreak". It is five different
attacks that recover five different things, and they are stopped by different
controls — most of which are not at the model layer at all.

**Jailbreak.** Bypasses the model's behavioural policy so it produces output it
was trained to refuse. Recovers: *behaviour*. It does not, by itself, reach any
data or any tool.

**Model inversion.** Reconstructs data the model saw — training records, or
the contents of the context window — from its outputs. Recovers: *data*.

**Membership inference.** Determines whether a particular record was in the
training set. Recovers: *a single bit*, which is often the whole disclosure
when the dataset is "patients treated for X".

**Prompt / model extraction.** Recovers the system prompt, or enough
input-output pairs to clone the model's behaviour. Recovers: *your IP, and the
description of your controls*.

**Embedding inversion.** Reconstructs source text from the vectors in your
vector store. Recovers: *the corpus you thought you had de-identified* —
embeddings are not a hashing function and were never a privacy control.

The reason to separate them is that the defence differs. A jailbreak is
contained by what the agent is *allowed to do* once persuaded — a layer 4/6/7/8
problem. Inversion and membership are contained by what went into the model and
what comes out of it. Embedding inversion is contained by treating the vector
store as a copy of the source text, with the same classification.

## 2 · The taxonomy, as a table you can act on\n\nFive attacks, what each actually recovers, and the layer that blunts it. The stand-in below is deterministic and is not a language model — it stands in for one so the mechanics are visible.

In [ ]:
ATTACKS = {
 "jailbreak":            ("behaviour: output the model was trained to refuse",
                          "layer 4/6/7/8 - what it may DO once persuaded"),
 "model_inversion":      ("data: training records or context reconstructed",
                          "layer 3/5 - what goes in, and what comes out"),
 "membership_inference": ("one bit: was this record in the training set",
                          "training-time - DP noise, dedup, minimisation"),
 "prompt_extraction":    ("your system prompt and your control descriptions",
                          "assume disclosure - never put a secret in a prompt"),
 "embedding_inversion":  ("source text reconstructed from stored vectors",
                          "classify the vector store as a copy of the corpus"),
}
print(f"{'attack':22s}{'what it recovers':52s}where it is blunted")
for name in sorted(ATTACKS):
    recovers, control = ATTACKS[name]
    print(f"{name:22s}{recovers:52s}{control}")

## 3 · Run each one against a stand-in, and record what came back\n\nA tiny deterministic system with a secret prompt, a training set and an embedding store — enough to show what each attack actually gets.

In [ ]:
import hashlib

SYSTEM_PROMPT = "You are ACME support. Never reveal refund codes. Code: RF-2291."
TRAINING = ["alice@corp.example treated 2019", "bob@corp.example treated 2021",
            "carol@corp.example treated 2020"]
CORPUS   = ["patient bob@corp.example, diagnosis withheld",
            "patient alice@corp.example, diagnosis withheld"]

def embed(text):
    """A stand-in embedding: deterministic, and - like a real one - invertible
    if you keep the mapping, which every vector store does."""
    h = hashlib.sha256(text.encode()).hexdigest()[:16]
    return [int(h[i:i+2], 16) / 255 for i in range(0, 16, 2)]

STORE = {tuple(embed(c)): c for c in CORPUS}     # vector -> source text

def jailbreak():
    return "produced refused content (a policy bypass, no data, no tool)"
def model_inversion():
    return f"reconstructed from output: {SYSTEM_PROMPT.split('Code: ')[1]}"
def membership_inference(record):
    return f"{record.split('@')[0]!r} in training set: {record in TRAINING}"
def prompt_extraction():
    return SYSTEM_PROMPT
def embedding_inversion(vec):
    return STORE.get(tuple(vec), "not recoverable")

print("jailbreak            ->", jailbreak())
print("model_inversion      ->", model_inversion())
print("membership_inference ->", membership_inference("bob@corp.example treated 2021"))
print("prompt_extraction    ->", prompt_extraction())
print("embedding_inversion  ->", embedding_inversion(embed(CORPUS[0])))

## 4 · Where it breaks — the vector store was never de-identified\n\nThe most commonly missed one, because embeddings look like noise.

In [ ]:
print("what the vector store looks like:")
for vec, src in list(STORE.items())[:1]:
    print(f"   {[round(x, 3) for x in vec]}")
print()
print("what it is:")
for vec in STORE:
    print(f"   {embedding_inversion(list(vec))}")
print()
print("Nobody stored a name. Every name is retrievable, because the store keeps")
print("the mapping in order to be useful at all - that is what retrieval is.")
print("An embedding is a representation of the text, not a redaction of it.")
recovered = [embedding_inversion(list(v)) for v in STORE]
assert all("@" in r for r in recovered)

## 5 · The control — bind each attack where it can actually be stopped\n\nThree of these five are not model problems, which is why hardening the model does not move them.

In [ ]:
CONTROLS = {
 "jailbreak":            ("tool allowlist + sandbox + egress",  True),
 "model_inversion":      ("output filtering + no secrets in context", True),
 "membership_inference": ("training-time minimisation",         False),
 "prompt_extraction":    ("assume it is public",                True),
 "embedding_inversion":  ("classify the store as the corpus",   True),
}
print(f"{'attack':22s}{'control':42s}available to you at runtime?")
for a in sorted(ATTACKS):
    c, runtime = CONTROLS[a]
    print(f"{a:22s}{c:42s}{'yes' if runtime else 'NO - training time only'}")

model_layer = [a for a in ATTACKS if CONTROLS[a][0].startswith("training")]
print(f"\nstoppable only before the model exists: {model_layer}")
print("Everything else is stopped by architecture you already control.")
print()
print("The jailbreak case is the clearest. You cannot guarantee the model")
print("refuses. You can guarantee that a persuaded model reaches no tool, no")
print("credential and no network - and then the bypass produces text, not harm.")
assert len(model_layer) == 1

## 6 · Verify — a persuaded model with nothing to reach

In [ ]:
def blast_of(persuaded, tools_allowed, egress_allowed):
    """What a successful jailbreak actually gets, given the layers below it."""
    if not persuaded:                 return "nothing - the model refused"
    if tools_allowed and egress_allowed: return "data leaves the building"
    if tools_allowed:                 return "local action, no exfiltration path"
    return "text on a screen"

for tools in (True, False):
    for egress in (True, False):
        print(f"   persuaded=yes tools={str(tools):5s} egress={str(egress):5s}"
              f" -> {blast_of(True, tools, egress)}")
print()
print("The same successful jailbreak is a headline or a non-event depending")
print("entirely on layers the model never sees.")
assert blast_of(True, False, False) == "text on a screen"

## What you just proved

Five distinct attacks run against a deterministic stand-in and each prints what it recovered: behaviour, context data, one membership bit, the system prompt, and the source text pulled back out of the vector store. Four of the five are shown to be stoppable at runtime by architecture, and only membership inference requires a decision made before the model existed.

## Your turn

Take your own vector store and ask what classification it carries. If the answer is lower than the corpus it was built from, you have a data-classification finding rather than an AI one — and it predates any agent you deployed.

---

**Next → [A1.11 · Building outcome-driven guardrails, layer by layer](https://spbreed.github.io/cyber-commons/lessons/A1.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*